# 00 · 数据

把各篇研究用到的行情取到本地 SQLite 数据库 `data/tailrisk.db`，研究 notebook 只从这个库里读。

- 全部来源免账号：中债指数直接请求中国债券信息网；A 股 / ETF / 港美股经 AkShare（新浪、腾讯、东方财富）；
  美国利率与波动率指数来自 FRED；美股因子来自 Kenneth French 数据库。
- 已经在库里的序列默认跳过；把 `REFRESH` 改成 `True` 会全部重新下载。新增标的只需在 `SERIES` 里加一行。
- ⚠ **数据库不入库**（已 gitignore）：中债等数据不允许再分发，仓库里只放取数代码。

**表结构**

| 表 | 内容 |
|---|---|
| `prices(name, date, value)` | 日频收盘价 / 财富指数 / 利率，主键 (name, date) |
| `french_factors(date, factor, value)` | 月度因子收益（小数），主键 (date, factor) |
| `series_info(name, source, arg, first_date, last_date, n, updated_at)` | 每条序列的来源与更新时间 |

In [1]:
import io
import sqlite3
import time
import zipfile
from contextlib import closing
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

REFRESH = False      # True：已在库里的序列也重新下载

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "01_methods.ipynb").exists())
DB_PATH = ROOT / "data" / "tailrisk.db"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

SCHEMA = """
CREATE TABLE IF NOT EXISTS prices (
    name TEXT, date TEXT, value REAL, PRIMARY KEY (name, date));
CREATE TABLE IF NOT EXISTS french_factors (
    date TEXT, factor TEXT, value REAL, PRIMARY KEY (date, factor));
CREATE TABLE IF NOT EXISTS series_info (
    name TEXT PRIMARY KEY, source TEXT, arg TEXT,
    first_date TEXT, last_date TEXT, n INTEGER, updated_at TEXT);
"""
with closing(sqlite3.connect(DB_PATH)) as con:
    con.executescript(SCHEMA)

## 标的登记

In [2]:
# 名称 → (来源, 参数)。新增标的只需在这里加一行。
SERIES = {
    "沪深300":         ("ak_index", "sh000300"),
    "中证500":         ("ak_index", "sh000905"),
    "上证50":          ("ak_index", "sh000016"),
    "上证国债指数":     ("ak_index", "sh000012"),
    "上证企业债指数":   ("ak_index", "sh000013"),
    "中债综合财富":     ("chinabond", "2c90818811afed8d0111c0c672b31578"),  # 中金原文所用
    "中债新综合财富":   ("chinabond", "8a8b2ca0332abed20134ea76d8885831"),
    "中债国债总财富":   ("chinabond", "2c9081e50e8767dc010e879acb220021"),  # 国泰君安（2018）所用
    "黄金Au9999":      ("ak_sge", "Au99.99"),       # 上海金交所现货，免费数据仅 2016-12 起
    "黄金ETF":         ("ak_etf_hfq", "518880"),     # 华安黄金ETF（跟踪 Au99.99，后复权），2013-07 起；与 A 股同时段交易
    # 东北证券（2025）示例中的 10 只 ETF，均为后复权（含分红再投资）
    "沪深300ETF":      ("ak_etf_hfq", "510300"),
    "中证500ETF":      ("ak_etf_hfq", "159922"),
    "中证1000ETF":     ("ak_etf_hfq", "512100"),
    "创业板ETF":       ("ak_etf_hfq", "159908"),
    "消费ETF":         ("ak_etf_hfq", "159928"),
    "纳指ETF":         ("ak_etf_hfq", "513100"),
    "恒生ETF":         ("ak_etf_hfq", "159920"),
    "红利ETF":         ("ak_etf_hfq", "510880"),
    "大宗商品ETF":     ("ak_etf_hfq", "510170"),
    "标普500ETF":      ("ak_etf_hfq", "513500"),     # 国泰君安（2018）五资产示例中标普500 的同步替代
    # 国泰君安（2018）复现用的长历史序列（2006 年起）
    "标普500":         ("ak_us_index", ".INX"),
    "恒生指数(盈富基金)": ("ak_hk_hfq", "02800"),       # 新浪恒指仅 2013 年起；盈富基金与恒指日收益相关 0.98
    "黄金(GLD×美元兑人民币)": ("gold_cny_gld", None),  # 免费的上金所 Au99.99 仅 2016-12 起
    # 兴业（2025）单资产尾部预警所需
    "GLD":             ("ak_us_daily", "GLD"),
    "纳斯达克100":      ("fred", "NASDAQ100"),
    "布伦特原油":       ("fred", "DCOILBRENTEU"),
    "美国10年实际利率":  ("fred", "DFII10"),
    "VIX":             ("fred", "VIXCLS"),
    "VXN":             ("fred", "VXNCLS"),
    "OVX":             ("fred", "OVXCLS"),
    "美元指数(美联储发达经济体)": ("fred", "DTWEXAFEGS"),   # ICE 美元指数无免费数据，此为构成接近的替代
    "美元兑人民币":     ("fred", "DEXCHUS"),               # 离岸人民币无免费长历史，用在岸替代
    "上证A股股息率":    ("lg_gxl", "上证A股"),
    "中国10年国债收益率": ("cn_10y", None),
}


# 申万一级行业（31 个），用于以行业代替成分股计算 PcaScore
SW_L1 = ["801010", "801030", "801040", "801050", "801080", "801110", "801120", "801130", "801140",
         "801150", "801160", "801170", "801180", "801200", "801210", "801230", "801710", "801720",
         "801730", "801740", "801750", "801760", "801770", "801780", "801790", "801880", "801890",
         "801950", "801960", "801970", "801980"]

for code in SW_L1:
    SERIES[f"申万_{code}"] = ("sw_index", code)

## 取数

In [3]:
_CHINABOND_URL = "https://yield.chinabond.com.cn/cbweb-mn/indices/singleIndexQuery"


def _fetch_chinabond(index_id):
    """
    中国债券信息网单指数查询（财富指数、总值）。

    不用 AkShare 的 bond_composite_index_cbond：它把毫秒时间戳按 UTC 转日期，
    北京时间零点变成前一天，整条序列错位一天（与股票收益的相关性因此被打乱）。
    这里按 Asia/Shanghai 转换。
    """
    params = {"indexid": index_id, "qxlxt": "00", "zslxt": "CFZS", "lx": "1", "locale": ""}
    js = requests.post(_CHINABOND_URL, params=params, timeout=30).json()["CFZS_00"]
    ts = pd.to_datetime(pd.Index(js.keys()).astype(float), unit="ms", utc=True)
    idx = ts.tz_convert("Asia/Shanghai").tz_localize(None).normalize()
    return pd.Series(pd.to_numeric(list(js.values()), errors="coerce"), index=idx)


_FRED_CSV = "https://fred.stlouisfed.org/graph/fredgraph.csv?id={}"


def _fetch_fred(series_id):
    """圣路易斯联储 FRED 公开数据（CSV，无需账号）。"""
    df = pd.read_csv(_FRED_CSV.format(series_id), index_col=0, parse_dates=True)
    return pd.to_numeric(df.iloc[:, 0], errors="coerce").dropna()

In [4]:
def _fetch(source, arg):
    if source == "chinabond":
        return _fetch_chinabond(arg)
    if source == "fred":
        return _fetch_fred(arg)
    if source == "gold_cny_gld":
        # SPDR 黄金 ETF（美元）× 美元兑人民币（FRED DEXCHUS），按美股交易日对齐、汇率前向填充
        gld = _fetch("ak_us_daily", "GLD")
        fx = _fetch_fred("DEXCHUS").reindex(gld.index, method="ffill")
        return (gld * fx).dropna()

    import akshare as ak

    if source == "ak_index":
        df = ak.stock_zh_index_daily(symbol=arg)
        return pd.Series(df["close"].values, index=pd.to_datetime(df["date"]))
    if source == "ak_etf_hfq":
        # 东方财富后复权；该接口连续请求易被限流，失败时改用腾讯后复权
        # （2016 年以来与东方财富的日收益差异 < 0.08%，来自价格三位小数的舍入）
        try:
            df = ak.fund_etf_hist_em(symbol=arg, period="daily", adjust="hfq")
            return pd.Series(df["收盘"].values, index=pd.to_datetime(df["日期"]))
        except Exception:
            prefix = "sh" if arg.startswith("5") else "sz"
            df = ak.stock_zh_a_hist_tx(symbol=prefix + arg, start_date="20000101",
                                       end_date=pd.Timestamp.today().strftime("%Y%m%d"), adjust="hfq")
            return pd.Series(df["close"].values, index=pd.to_datetime(df["date"]))
    if source == "lg_gxl":
        df = ak.stock_a_gxl_lg(symbol=arg)
        return pd.Series(df["股息率"].values, index=pd.to_datetime(df["日期"]))
    if source == "cn_10y":
        df = ak.bond_zh_us_rate(start_date="20020101")
        return pd.Series(df["中国国债收益率10年"].values, index=pd.to_datetime(df["日期"])).dropna()
    if source == "sw_index":
        df = ak.index_hist_sw(symbol=arg, period="day")
        return pd.Series(df["收盘"].values, index=pd.to_datetime(df["日期"]))
    if source == "ak_us_index":
        df = ak.index_us_stock_sina(symbol=arg)
        return pd.Series(df["close"].values, index=pd.to_datetime(df["date"]))
    if source == "ak_us_daily":
        df = ak.stock_us_daily(symbol=arg, adjust="")
        return pd.Series(df["close"].values, index=pd.to_datetime(df["date"]))
    if source == "ak_hk_hfq":
        df = ak.stock_hk_daily(symbol=arg, adjust="hfq")
        return pd.Series(df["close"].values, index=pd.to_datetime(df["date"]))
    if source == "ak_sge":
        df = ak.spot_hist_sge(symbol=arg)
        return pd.Series(df["close"].values, index=pd.to_datetime(df["date"]))
    raise ValueError(f"未知数据源 {source}")


def _fetch_with_retry(source, arg, tries=4):
    """部分免费接口（如东方财富）连续请求过快会断开连接：间隔递增地重试几次。"""
    for k in range(tries):
        try:
            return _fetch(source, arg)
        except Exception:
            if k == tries - 1:
                raise
            time.sleep(10 * 2 ** k)

In [5]:
_FRENCH = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{}_CSV.zip"


FRENCH_FILES = {
    "F-F_Research_Data_5_Factors_2x3": ["Mkt-RF", "SMB", "HML", "RMW", "CMA"],
    "F-F_Momentum_Factor": ["Mom"],
    "F-F_ST_Reversal_Factor": ["ST_Rev"],
    "F-F_LT_Reversal_Factor": ["LT_Rev"],
}


def _parse_french_monthly(text):
    """取 CSV 中第一段月度数据（行首为 YYYYMM），到第一个空行或年度段为止。数值为百分比。"""
    lines = text.splitlines()
    start = next(i for i, l in enumerate(lines) if l.strip().split(",")[0].strip().isdigit()
                 and len(l.strip().split(",")[0].strip()) == 6)
    header = lines[start - 1]
    body = []
    for l in lines[start:]:
        key = l.split(",")[0].strip()
        if not (key.isdigit() and len(key) == 6):
            break
        body.append(l)
    df = pd.read_csv(io.StringIO("\n".join([header] + body)), index_col=0)
    df.columns = [c.strip() for c in df.columns]
    df.index = pd.to_datetime(df.index.astype(str), format="%Y%m") + pd.offsets.MonthEnd(0)
    return df.apply(pd.to_numeric, errors="coerce")


def fetch_french():
    """八个美股因子的月度收益（小数）：Mkt-RF、SMB、HML、RMW、CMA、Mom、ST_Rev、LT_Rev。"""
    parts = []
    for name, cols in FRENCH_FILES.items():
        raw = requests.get(_FRENCH.format(name), timeout=60).content
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            text = z.read(z.namelist()[0]).decode("latin-1")
        df = _parse_french_monthly(text)
        df.columns = [c.replace("Mom   ", "Mom").strip() for c in df.columns]
        rename = {df.columns[i]: cols[i] for i in range(len(cols))} if name != "F-F_Research_Data_5_Factors_2x3" else {}
        df = df.rename(columns=rename)
        parts.append(df[cols])
    return (pd.concat(parts, axis=1) / 100).dropna()

## 写入数据库

In [6]:
def in_db(name):
    with closing(sqlite3.connect(DB_PATH)) as con:
        return con.execute("SELECT 1 FROM series_info WHERE name = ?", (name,)).fetchone() is not None


def save_series(name, s):
    """整条替换一条序列：先删后写，放在同一个事务里。"""
    s = s.sort_index()
    s = s[~s.index.duplicated(keep="last")].astype(float).dropna()
    rows = [(name, d.strftime("%Y-%m-%d"), float(v)) for d, v in s.items()]
    source, arg = SERIES[name]
    with closing(sqlite3.connect(DB_PATH)) as con, con:
        con.execute("DELETE FROM prices WHERE name = ?", (name,))
        con.executemany("INSERT INTO prices VALUES (?, ?, ?)", rows)
        con.execute("INSERT OR REPLACE INTO series_info VALUES (?, ?, ?, ?, ?, ?, ?)",
                    (name, source, arg, rows[0][1], rows[-1][1], len(rows), datetime.now().isoformat(timespec="seconds")))


def save_french(df):
    rows = [(d.strftime("%Y-%m-%d"), f, float(v)) for d, r in df.iterrows() for f, v in r.items()]
    with closing(sqlite3.connect(DB_PATH)) as con, con:
        con.execute("DELETE FROM french_factors")
        con.executemany("INSERT INTO french_factors VALUES (?, ?, ?)", rows)
        con.execute("INSERT OR REPLACE INTO series_info VALUES (?, ?, ?, ?, ?, ?, ?)",
                    ("french_factors", "french", ",".join(df.columns), rows[0][0], rows[-1][0], len(df),
                     datetime.now().isoformat(timespec="seconds")))

In [7]:
failed = []
for name, (source, arg) in SERIES.items():
    if in_db(name) and not REFRESH:
        continue
    try:
        save_series(name, _fetch_with_retry(source, arg))
        print("已更新", name)
    except Exception as e:
        failed.append(name)
        print("失败", name, type(e).__name__, e)
    time.sleep(1)          # 免费接口连续请求过快会被限流

if REFRESH or not in_db("french_factors"):
    save_french(fetch_french())
    print("已更新 french_factors")

print("失败：", failed or "无")

失败： 无


## 库里现有的数据

In [8]:
with closing(sqlite3.connect(DB_PATH)) as con:
    info = pd.read_sql("SELECT * FROM series_info ORDER BY name", con, index_col="name")
info

,source,arg,first_date,last_date,n,updated_at
name,,,,,,
GLD,ak_us_daily,GLD,2004-11-18,2026-10-19,5512,2026-10-25T18:06:54
OVX,fred,OVXCLS,2007-05-10,2026-10-16,4893,2026-10-25T18:06:55
VIX,fred,VIXCLS,1990-01-02,2026-10-16,9297,2026-10-25T18:06:55
VXN,fred,VXNCLS,2001-02-02,2026-10-16,6465,2026-10-25T18:06:55
french_factors,french,"Mkt-RF,SMB,HML,RMW,CMA,Mom,ST_Rev,LT_Rev",1963-07-31,2026-08-31,758,2026-10-25T18:06:59
...,...,...,...,...,...,...
美元指数(美联储发达经济体),fred,DTWEXAFEGS,2006-01-02,2026-10-16,5212,2026-10-25T18:06:55
美国10年实际利率,fred,DFII10,2003-01-02,2026-10-16,5952,2026-10-25T18:06:54
黄金(GLD×美元兑人民币),gold_cny_gld,None,2004-11-18,2026-10-16,5511,2026-10-25T18:06:54
